# Data Generation — synthetic failed payments

Generates 200–400 synthetic failed payments using `RazorpayClient.simulate_payment_failure`,
labels each with the rule-based classifier's output as ground truth, and saves to CSV
for `train_classifier.ipynb` to consume.

In [12]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `import app.xxx` works from notebooks/

import random
import pandas as pd

from app.razorpay_client import RazorpayClient
from app.classifier import classify_failure_rule_based

random.seed(42)
client = RazorpayClient()


In [13]:
import os
from dotenv import load_dotenv

# If your file is named secret.env, load it explicitly
load_dotenv("secret.env")

print("Key ID:", os.getenv("RAZORPAY_KEY_ID"))  # quick check


Key ID: rzp_test_TV5ZXEeBKj7oCk


## 1. Generate synthetic payments across all scenarios

We deliberately generate an *uneven* distribution (some scenarios more common than others)
since real-world failure distributions aren't uniform either.

In [14]:
SCENARIO_WEIGHTS = {
    'timeout': 60,
    'insufficient_funds': 80,
    'invalid_card': 70,
    'expired_card': 40,
    'auth_failure': 50,
    'risk_block': 25,
}

METHODS = ['card', 'upi', 'netbanking', 'wallet']
AMOUNTS = [9900, 19900, 49900, 99900, 149900, 299900, 499900]

records = []
for scenario, count in SCENARIO_WEIGHTS.items():
    for _ in range(count):
        amount = random.choice(AMOUNTS)
        method = random.choice(METHODS)
        raw = client.simulate_payment_failure(scenario, amount=amount, method=method)
        records.append({
            'payment_id': raw['id'],
            'failure_code': raw['error_code'],
            'failure_description': raw['error_description'],
            'method': raw['method'],
            'amount': raw['amount'],
            'true_scenario': scenario,  # the scenario we asked simulate_payment_failure for
        })

df = pd.DataFrame(records)
print(f'Generated {len(df)} synthetic failed payments')
df.head()


Generated 325 synthetic failed payments


,payment_id,failure_code,failure_description,method,amount,true_scenario
0,pay_SIMc3147dd40a0648,GATEWAY_ERROR,The card issuing bank server timed out.,card,299900,timeout
1,pay_SIM2bbf6da841d14f,GATEWAY_ERROR,The card issuing bank server timed out.,netbanking,9900,timeout
2,pay_SIMe0d593a3040149,GATEWAY_ERROR,The card issuing bank server timed out.,upi,19900,timeout
3,pay_SIMa18ce52abd454c,GATEWAY_ERROR,The card issuing bank server timed out.,card,19900,timeout
4,pay_SIM23ea5d01fd5742,GATEWAY_ERROR,The card issuing bank server timed out.,card,299900,timeout


## 2. Label with the rule-based classifier (ground truth)

We use the rule-based classifier's output as the ground-truth label — since it's
deterministic and directly derived from the failure_description, it's a reliable label
source for training. We also keep `true_scenario` (what we originally asked for) to sanity
check the rule-based classifier isn't misfiring.

In [15]:
class Row:
    def __init__(self, code, desc):
        self.failure_code = code
        self.failure_description = desc

df['root_cause'] = df.apply(
    lambda r: classify_failure_rule_based(Row(r['failure_code'], r['failure_description'])),
    axis=1
)

# Sanity check: rule-based label should match the scenario we asked for
mismatch = df[df['root_cause'] != df['true_scenario']]
print(f'Rule-based vs requested-scenario mismatches: {len(mismatch)} / {len(df)}')
mismatch


Rule-based vs requested-scenario mismatches: 0 / 325


,payment_id,failure_code,failure_description,method,amount,true_scenario,root_cause


## 3. Add a categorical `bank`-style noise column (optional realism)

Real Razorpay payments carry issuer/bank info that correlates loosely with failure type
(e.g. certain banks time out more). We simulate this lightly so the ML model in the next
notebook has more than 2 features to work with.

In [16]:
BANKS = ['HDFC', 'ICICI', 'SBI', 'AXIS', 'KOTAK']
df['bank'] = [random.choice(BANKS) for _ in range(len(df))]

df.to_csv('synthetic_failed_payments.csv', index=False)
print('Saved notebooks/synthetic_failed_payments.csv')
df['root_cause'].value_counts()


Saved notebooks/synthetic_failed_payments.csv


root_cause
insufficient_funds    80
invalid_card          70
timeout               60
auth_failure          50
expired_card          40
risk_block            25
Name: count, dtype: int64

In [17]:
import os
import sys
import random
import pandas as pd
from dotenv import load_dotenv

# Ensure imports from app/ work
sys.path.append(os.path.abspath(".."))

# Load environment variables (adjust filename if needed)
load_dotenv("secret.env")  # or just load_dotenv() if you renamed to .env

from app.razorpay_client import RazorpayClient
from app.classifier import classify_failure_rule_based

# Fix random seed for reproducibility
random.seed(42)

# Initialize client
client = RazorpayClient()

# --- Synthetic data generation ---
rows = []
for i in range(325):  # same number as nbconvert run
    payment_id = f"pay_{i:04d}"
    failure_reason = random.choice(["network_error", "insufficient_funds", "card_declined"])
    label = classify_failure_rule_based(failure_reason)
    rows.append({"payment_id": payment_id, "failure_reason": failure_reason, "label": label})

df = pd.DataFrame(rows)

# Save CSV in the same folder as the notebook
output_path = "synthetic_failed_payments.csv"
df.to_csv(output_path, index=False)

print(f"✅ Generated {len(df)} labeled rows at {output_path}")


✅ Generated 325 labeled rows at synthetic_failed_payments.csv


In [1]:
import sys, os, random
import pandas as pd

# --- Locate project root (folder containing "app/") no matter where Jupyter's cwd is ---
def find_project_root(start=None):
    path = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, "app")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    raise RuntimeError(
        f"Could not find a folder containing 'app/' starting from {os.getcwd()}. "
        "Make sure you're running this notebook from inside razorpay-recovery-agent/notebooks/"
    )

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

# classify_failure_rule_based only needs app/models.py + app/classifier.py —
# it does NOT touch Razorpay API keys, so this works even without a valid .env
from app.classifier import classify_failure_rule_based

# --- Same test-failure scenarios as app/razorpay_client.py, inlined here so this
#     cell has zero dependency on RazorpayClient() (which requires real API keys) ---
TEST_FAILURE_SCENARIOS = {
    "timeout":             {"failure_code": "GATEWAY_ERROR",     "failure_description": "The card issuing bank server timed out."},
    "insufficient_funds":  {"failure_code": "BAD_REQUEST_ERROR", "failure_description": "Insufficient funds in the account."},
    "invalid_card":        {"failure_code": "BAD_REQUEST_ERROR", "failure_description": "The card number is invalid."},
    "expired_card":        {"failure_code": "BAD_REQUEST_ERROR", "failure_description": "The card has expired."},
    "auth_failure":        {"failure_code": "GATEWAY_ERROR",     "failure_description": "3D Secure authentication failed."},
    "risk_block":          {"failure_code": "GATEWAY_ERROR",     "failure_description": "Payment flagged by risk engine."},
}

SCENARIO_WEIGHTS = {
    "timeout": 60, "insufficient_funds": 80, "invalid_card": 70,
    "expired_card": 40, "auth_failure": 50, "risk_block": 25,
}
METHODS = ["card", "upi", "netbanking", "wallet"]
AMOUNTS = [9900, 19900, 49900, 99900, 149900, 299900, 499900]
BANKS = ["HDFC", "ICICI", "SBI", "AXIS", "KOTAK"]

random.seed(42)

class Row:  # tiny stand-in so classify_failure_rule_based(payment) works without an ORM object
    def __init__(self, code, desc):
        self.failure_code = code
        self.failure_description = desc

records = []
for scenario, count in SCENARIO_WEIGHTS.items():
    spec = TEST_FAILURE_SCENARIOS[scenario]
    for i in range(count):
        records.append({
            "payment_id": f"pay_SIM{scenario}_{i:04d}",
            "failure_code": spec["failure_code"],
            "failure_description": spec["failure_description"],
            "method": random.choice(METHODS),
            "amount": random.choice(AMOUNTS),
            "bank": random.choice(BANKS),
            "true_scenario": scenario,
        })

df = pd.DataFrame(records)

# label with the rule-based classifier as ground truth
df["root_cause"] = df.apply(
    lambda r: classify_failure_rule_based(Row(r["failure_code"], r["failure_description"])),
    axis=1,
)

# sanity check
mismatch = df[df["root_cause"] != df["true_scenario"]]
print(f"Generated {len(df)} rows. Mismatches vs requested scenario: {len(mismatch)}")

out_path = os.path.join(PROJECT_ROOT, "notebooks", "synthetic_failed_payments.csv")
df.to_csv(out_path, index=False)
print("Saved:", out_path)
df["root_cause"].value_counts()

Project root: C:\Users\Yazhini\Desktop\PayMend\payment-recovery
Generated 325 rows. Mismatches vs requested scenario: 0
Saved: C:\Users\Yazhini\Desktop\PayMend\payment-recovery\notebooks\synthetic_failed_payments.csv


root_cause
insufficient_funds    80
invalid_card          70
timeout               60
auth_failure          50
expired_card          40
risk_block            25
Name: count, dtype: int64